# ML_training — Prévision TempRet / PuisCpt par EGID

Exploration **multi-modèles** (Random Forest, XGBoost, LSTM) sur des séries au pas 15 min, par cluster.

**Prérequis :** ouvrir ce notebook avec le répertoire de travail = dossier `2_Program/` (chemins relatifs `0_Data/...`).

**Données :** `3_Training`, `4_Validation`, `5_Test` — fichiers `cluster{N}.parquet` (N ∈ {3,4,5,6}).

**Sorties :** modèles dans `0_Data/6_Models/Cluster{N}/` (dont `*_train_val_diagnostics.png` + `*_training_summary.json` par EGID) ; prédictions test dans `0_Data/9_Results/Cluster{N}/`.

Principes : split temporel déjà appliqué dans le pipeline ; **pas de fuite** (lags décalés, scaler fit sur train uniquement) ; **RAM** : chargement par EGID, purge après chaque modèle.

**Entrées / cibles** : uniquement des grandeurs **normalisées [0, 1]** comme en `dataset_preparation_V2` (section 6) : exogènes `TempExt_norm` (pas `TempExt`), cibles `TempRet_norm` et `PuisCpt_fc`. Les exports et graphiques **repassent `TempRet` en °C** (inverse de la norme) ; **PuisCpt** reste en facteur de charge fc.


## 1. Configuration

In [ ]:
from __future__ import annotations

import gc
import logging
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("ml_training")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------------------
# Chemins : cwd = 2_Program
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
PATH_TRAIN = NOTEBOOK_DIR / "0_Data" / "3_Training"
PATH_VAL = NOTEBOOK_DIR / "0_Data" / "4_Validation"
PATH_TEST = NOTEBOOK_DIR / "0_Data" / "5_Test"
PATH_MODELS = NOTEBOOK_DIR / "0_Data" / "6_Models"
PATH_RESULTS = NOTEBOOK_DIR / "0_Data" / "9_Results"

CLUSTER_IDS = [3, 4, 5, 6]
SEED = 42
# -1 ou valeur > nombre d'EGID du cluster → entraîner sur **tous** les EGID du cluster
N_EGID_PER_CLUSTER = -1

# Parallélisme RF + XGB (joblib / OpenMP) : n_jobs négatif → max(1, n_cpu + 1 + n_jobs)
N_JOBS_PARALLEL = -5

# Features dérivées (lags / rolling) — aligné skill time-series
LAGS = [1, 2, 3, 7]
ROLL_WINDOWS = [3, 7]

EXOG_COLS = [
    "TempExt_norm",
    "dayofyear_cos",
    "dayofyear_sin",
    "dayofweek_cos",
    "dayofweek_sin",
    "hour_cos",
    "hour_sin",
]

# Random Forest — grille réduite, sélection sur validation
RF_PARAM_GRID = [
    {"n_estimators": 80, "max_depth": 8},
    {"n_estimators": 120, "max_depth": 10},
    {"n_estimators": 150, "max_depth": 12},
    {"n_estimators": 120, "max_depth": None},
]

# XGBoost
XGB_PARAMS = dict(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=N_JOBS_PARALLEL,
    verbosity=0,
)
XGB_EARLY_STOPPING_ROUNDS = 40

# LSTM
SEQ_LEN = 24
LSTM_EPOCHS = 80
LSTM_BATCH_SIZE = 64
LSTM_PATIENCE = 12
LSTM_ADAM_LEARNING_RATE = 1e-3  # optimiseur Adam (compile LSTM)
LSTM_COMPILE_LOSS = "mse"

# Dénormalisation TempRet (°C) — aligné pipeline : clip((T-20)/60, 0, 1)
TEMPRET_NORM_T0_C = 20.0
TEMPRET_NORM_SCALE_C = 60.0

RNG = np.random.default_rng(SEED)

# Graphiques section analyse
PLOT_START = None  # ex. pd.Timestamp("2025-01-01", tz="UTC") ou None = auto (derniers points)
PLOT_END = None
PLOT_MAX_POINTS = 500  # si pas de plage explicite : tracer les N derniers pas temporels valides

SELECTED_EGIDS: dict[int, list[str]] = {}


In [ ]:
import tensorflow as tf

tf.keras.utils.set_random_seed(SEED)


def parse_egids(columns: list[str]) -> list[str]:
    """EGID à partir des colonnes *.TempRet_norm."""
    out: set[str] = set()
    for c in columns:
        if c.endswith(".TempRet_norm"):
            out.add(c[: -len(".TempRet_norm")])
    return sorted(out)


def columns_for_egid(egid: str) -> list[str]:
    eg = str(egid)
    return (
        ["Dates"]
        + EXOG_COLS
        + [
            f"{eg}.TempRet_norm",
            f"{eg}.PuisCpt_fc",
            f"{eg}.TempRet.inv",
            f"{eg}.PuisCpt.inv",
        ]
    )


def feature_column_names() -> list[str]:
    feats = list(EXOG_COLS)
    for lag in LAGS:
        feats += [f"lag_tr_{lag}", f"lag_pc_{lag}"]
    for w in ROLL_WINDOWS:
        feats += [
            f"roll_tr_mean_{w}",
            f"roll_tr_std_{w}",
            f"roll_pc_mean_{w}",
            f"roll_pc_std_{w}",
        ]
    return feats


FEATURE_COLS = feature_column_names()


def add_lag_features(df: pd.DataFrame, egid: str) -> pd.DataFrame:
    eg = str(egid)
    tr, pc = f"{eg}.TempRet_norm", f"{eg}.PuisCpt_fc"
    out = df.copy()
    for lag in LAGS:
        out[f"lag_tr_{lag}"] = out[tr].shift(lag)
        out[f"lag_pc_{lag}"] = out[pc].shift(lag)
    shifted_tr = out[tr].shift(1)
    shifted_pc = out[pc].shift(1)
    for w in ROLL_WINDOWS:
        out[f"roll_tr_mean_{w}"] = shifted_tr.rolling(w).mean()
        out[f"roll_tr_std_{w}"] = shifted_tr.rolling(w).std()
        out[f"roll_pc_mean_{w}"] = shifted_pc.rolling(w).mean()
        out[f"roll_pc_std_{w}"] = shifted_pc.rolling(w).std()
    return out


def free_ram(*objs) -> None:
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()


def free_tf() -> None:
    try:
        tf.keras.backend.clear_session()
    except Exception as e:
        logger.warning("clear_session: %s", e)
    gc.collect()


def load_concat_frames(cluster_id: int, egid: str) -> tuple[pd.DataFrame, tuple[int, int, int]]:
    """Charge train/val/test (colonnes minimales), concatène avec marqueur _sp."""
    cols = columns_for_egid(egid)
    tr_path = PATH_TRAIN / f"cluster{cluster_id}.parquet"
    va_path = PATH_VAL / f"cluster{cluster_id}.parquet"
    te_path = PATH_TEST / f"cluster{cluster_id}.parquet"
    dtr = pd.read_parquet(tr_path, columns=cols)
    dva = pd.read_parquet(va_path, columns=cols)
    dte = pd.read_parquet(te_path, columns=cols)
    n_tr, n_va, n_te = len(dtr), len(dva), len(dte)
    dtr = dtr.copy()
    dva = dva.copy()
    dte = dte.copy()
    dtr["_sp"] = 0
    dva["_sp"] = 1
    dte["_sp"] = 2
    full = pd.concat([dtr, dva, dte], ignore_index=True)
    full = add_lag_features(full, egid)
    free_ram(dtr, dva, dte)
    return full, (n_tr, n_va, n_te)


def build_xy_matrices(full: pd.DataFrame, egid: str):
    """Retourne X_raw, y ([TempRet_norm, PuisCpt_fc] dans [0,1]), sp, inv_ok, dates."""
    eg = str(egid)
    tr_c = f"{eg}.TempRet_norm"
    pc_c = f"{eg}.PuisCpt_fc"
    vi_tr = f"{eg}.TempRet.inv"
    vi_pc = f"{eg}.PuisCpt.inv"

    feat_ok = full[FEATURE_COLS].replace([np.inf, -np.inf], np.nan).notna().all(axis=1)
    y_ok = full[[tr_c, pc_c]].replace([np.inf, -np.inf], np.nan).notna().all(axis=1)
    ready = feat_ok & y_ok

    sub = full.loc[ready].reset_index(drop=True)
    X_raw = sub[FEATURE_COLS].to_numpy(dtype=np.float64)
    y = np.clip(sub[[tr_c, pc_c]].to_numpy(dtype=np.float64), 0.0, 1.0)
    sp = sub["_sp"].to_numpy(dtype=np.int8)
    dates = sub["Dates"].to_numpy()
    inv_ok = (sub[vi_tr].to_numpy() == 0) & (sub[vi_pc].to_numpy() == 0)

    return X_raw, y, sp, inv_ok, dates, sub[[vi_tr, vi_pc]]


def rmse_per_target(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    return np.sqrt(mean_squared_error(y_true, y_pred, multioutput="raw_values"))


def aggregate_score(rmses: np.ndarray) -> float:
    """Combine les deux cibles (moyenne des RMSE) pour comparaison unique."""
    return float(np.mean(rmses))


def ensure_dirs(cluster_id: int) -> tuple[Path, Path]:
    mdir = PATH_MODELS / f"Cluster{cluster_id}"
    rdir = PATH_RESULTS / f"Cluster{cluster_id}"
    mdir.mkdir(parents=True, exist_ok=True)
    rdir.mkdir(parents=True, exist_ok=True)
    return mdir, rdir


def denorm_tempret_celsius(norm: np.ndarray) -> np.ndarray:
    """Inverse norme pipeline : T(°C) = TEMPRET_NORM_T0_C + scale * clip(norm, 0, 1)."""
    x = np.clip(np.asarray(norm, dtype=np.float64), 0.0, 1.0)
    return TEMPRET_NORM_T0_C + TEMPRET_NORM_SCALE_C * x


def export_test_predictions(
    path_parquet: Path,
    dates: np.ndarray,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    inv_df: pd.DataFrame,
) -> None:
    yt = np.clip(np.asarray(y_true, dtype=np.float64), 0.0, 1.0)
    yp = np.clip(np.asarray(y_pred, dtype=np.float64), 0.0, 1.0)
    out = pd.DataFrame(
        {
            "Dates": dates,
            "TempRet": denorm_tempret_celsius(yt[:, 0]),
            "PuisCpt": yt[:, 1],
            "TempRetPred": denorm_tempret_celsius(yp[:, 0]),
            "PuisCptPred": yp[:, 1],
        }
    )
    if inv_df is not None and len(inv_df) == len(out):
        out["TempRet.inv"] = inv_df.iloc[:, 0].to_numpy()
        out["PuisCpt.inv"] = inv_df.iloc[:, 1].to_numpy()
    out.to_parquet(path_parquet, index=False)


def collect_lstm_end_indices(sp: np.ndarray, seq_len: int) -> tuple[list[int], list[int], list[int]]:
    """Indices de fin de fenêtre (inclus) pour train / val / test sans chevauchement de split."""
    n = len(sp)
    train_e, val_e, test_e = [], [], []
    for i in range(seq_len - 1, n):
        start = i - seq_len + 1
        seg = sp[start : i + 1]
        if seg[-1] == 0 and bool(np.all(seg == 0)):
            train_e.append(i)
        elif seg[-1] == 1 and int(seg.max()) <= 1:
            val_e.append(i)
        elif seg[-1] == 2:
            test_e.append(i)
    return train_e, val_e, test_e


def ends_to_seq_X(X: np.ndarray, ends: list[int], seq_len: int) -> np.ndarray:
    return np.stack([X[t - seq_len + 1 : t + 1] for t in ends], axis=0)


import json

import matplotlib.pyplot as plt


TARGET_METRIC_NAMES = ("TempRet_norm", "PuisCpt_fc")


def mae_rmse_per_target(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    yt = np.asarray(y_true, dtype=np.float64)
    yp = np.asarray(y_pred, dtype=np.float64)
    mae = np.array([mean_absolute_error(yt[:, j], yp[:, j]) for j in range(yt.shape[1])])
    rmse = np.sqrt(
        np.array([mean_squared_error(yt[:, j], yp[:, j]) for j in range(yt.shape[1])])
    )
    return mae, rmse


def _multioutput_rf_feature_importances(rf) -> np.ndarray:
    n_out = getattr(rf, "n_outputs_", 1)
    est = getattr(rf, "estimators_", None)
    if est is not None and len(est) == n_out and n_out > 1:
        return np.mean(
            [np.asarray(e.feature_importances_, dtype=np.float64) for e in est],
            axis=0,
        )
    return np.asarray(rf.feature_importances_, dtype=np.float64)


def plot_rf_diagnostics(
    model_dir: Path,
    prefix: str,
    egid: str,
    cluster_id: int,
    y_tr: np.ndarray,
    pred_tr: np.ndarray,
    y_va: np.ndarray,
    pred_va: np.ndarray,
    feature_names: list[str],
    rf_model,
    best_params: dict,
    grid_results: list[tuple[dict, float]],
) -> None:
    mae_tr, rmse_tr = mae_rmse_per_target(y_tr, pred_tr)
    mae_va, rmse_va = mae_rmse_per_target(y_va, pred_va)
    imp = _multioutput_rf_feature_importances(rf_model)
    top_i = np.argsort(imp)[-5:][::-1]
    top_names = [feature_names[i] for i in top_i]
    top_vals = imp[top_i]

    fig = plt.figure(figsize=(11, 8))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.1, 1.0])
    ax0 = fig.add_subplot(gs[0, 0])
    x = np.arange(len(TARGET_METRIC_NAMES))
    w = 0.18
    ax0.bar(x - 1.5 * w, mae_tr, w, label="MAE train")
    ax0.bar(x - 0.5 * w, mae_va, w, label="MAE val")
    ax0.bar(x + 0.5 * w, rmse_tr, w, label="RMSE train")
    ax0.bar(x + 1.5 * w, rmse_va, w, label="RMSE val")
    ax0.set_xticks(x)
    ax0.set_xticklabels(TARGET_METRIC_NAMES)
    ax0.set_ylabel("Erreur (cibles norm. [0,1])")
    ax0.legend(fontsize=7)
    ax0.set_title("Train vs validation")
    ax0.grid(axis="y", alpha=0.3)

    ax1 = fig.add_subplot(gs[0, 1])
    ax1.barh(top_names[::-1], top_vals[::-1], color="steelblue")
    ax1.set_title("Top 5 features (importance moyenne sur les 2 cibles)")
    ax1.grid(axis="x", alpha=0.3)

    ax2 = fig.add_subplot(gs[1, :])
    ax2.axis("off")
    lines = [
        f"Cluster {cluster_id} — EGID {egid}",
        "",
        "Hyperparamètres retenus (grille RF_PARAM_GRID):",
        json.dumps(best_params, indent=2, default=str),
        "",
        "Scores validation (RMSE moyen des 2 cibles) par jeu de paramètres:",
    ]
    for pr, sc in grid_results:
        lines.append(f"  {sc:.6f}  <-  {pr}")
    ax2.text(0, 1, "\n".join(lines), transform=ax2.transAxes, va="top", fontsize=8, family="monospace")

    fig.suptitle(f"{prefix} — diagnostic entraînement / validation")
    fig.tight_layout()
    fig.savefig(model_dir / f"{prefix}_{egid}_train_val_diagnostics.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    summary = {
        "best_params": best_params,
        "grid_search_val_mean_rmse": [{"params": dict(p), "val_mean_rmse": s} for p, s in grid_results],
        "train_mae": mae_tr.tolist(),
        "val_mae": mae_va.tolist(),
        "train_rmse": rmse_tr.tolist(),
        "val_rmse": rmse_va.tolist(),
        "top5_features": [{"name": n, "importance": float(v)} for n, v in zip(top_names, top_vals)],
    }
    (model_dir / f"{prefix}_{egid}_training_summary.json").write_text(
        json.dumps(summary, indent=2, default=str), encoding="utf-8"
    )


def plot_xgb_diagnostics(
    model_dir: Path,
    prefix: str,
    egid: str,
    cluster_id: int,
    y_tr: np.ndarray,
    pred_tr: np.ndarray,
    y_va: np.ndarray,
    pred_va: np.ndarray,
    feature_names: list[str],
    models_dict: dict,
) -> None:
    mae_tr, rmse_tr = mae_rmse_per_target(y_tr, pred_tr)
    mae_va, rmse_va = mae_rmse_per_target(y_va, pred_va)
    imp_a = np.asarray(models_dict["TempRet"].feature_importances_, dtype=np.float64)
    imp_b = np.asarray(models_dict["PuisCpt"].feature_importances_, dtype=np.float64)
    imp_mean = (imp_a + imp_b) / 2.0
    top_i = np.argsort(imp_mean)[-5:][::-1]
    top_names = [feature_names[i] for i in top_i]
    top_vals = imp_mean[top_i]

    fig = plt.figure(figsize=(11, 7))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.1, 1.0])
    ax0 = fig.add_subplot(gs[0, 0])
    x = np.arange(len(TARGET_METRIC_NAMES))
    w = 0.18
    ax0.bar(x - 1.5 * w, mae_tr, w, label="MAE train")
    ax0.bar(x - 0.5 * w, mae_va, w, label="MAE val")
    ax0.bar(x + 0.5 * w, rmse_tr, w, label="RMSE train")
    ax0.bar(x + 1.5 * w, rmse_va, w, label="RMSE val")
    ax0.set_xticks(x)
    ax0.set_xticklabels(TARGET_METRIC_NAMES)
    ax0.set_ylabel("Erreur (cibles norm. [0,1])")
    ax0.legend(fontsize=7)
    ax0.set_title("Train vs validation (early stopping sur val)")
    ax0.grid(axis="y", alpha=0.3)

    ax1 = fig.add_subplot(gs[0, 1])
    ax1.barh(top_names[::-1], top_vals[::-1], color="darkorange")
    ax1.set_title("Top 5 features (gain moyen XGB, 2 cibles)")
    ax1.grid(axis="x", alpha=0.3)

    ax2 = fig.add_subplot(gs[1, :])
    ax2.axis("off")
    meta_lines = [f"Cluster {cluster_id} — EGID {egid}", "", "Early stopping / arbres:"]
    for name, m in models_dict.items():
        bi = getattr(m, "best_iteration", None)
        bi_s = int(bi) if bi is not None else None
        meta_lines.append(f"  {name}: best_iteration={bi_s}, n_estimators={int(m.n_estimators)}")
    ax2.text(0, 1, "\n".join(meta_lines), transform=ax2.transAxes, va="top", fontsize=9, family="monospace")

    fig.suptitle(f"{prefix} — diagnostic entraînement / validation")
    fig.tight_layout()
    fig.savefig(model_dir / f"{prefix}_{egid}_train_val_diagnostics.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    xmeta = {
        name: {
            "best_iteration": int(m.best_iteration) if getattr(m, "best_iteration", None) is not None else None,
            "n_estimators": int(m.n_estimators),
        }
        for name, m in models_dict.items()
    }
    xmeta["train_mae"] = mae_tr.tolist()
    xmeta["val_mae"] = mae_va.tolist()
    xmeta["train_rmse"] = rmse_tr.tolist()
    xmeta["val_rmse"] = rmse_va.tolist()
    xmeta["top5_features"] = [{"name": n, "importance_mean": float(v)} for n, v in zip(top_names, top_vals)]
    (model_dir / f"{prefix}_{egid}_training_summary.json").write_text(
        json.dumps(xmeta, indent=2, default=str), encoding="utf-8"
    )


def plot_lstm_diagnostics(
    model_dir: Path,
    prefix: str,
    egid: str,
    cluster_id: int,
    history,
    y_tr: np.ndarray,
    pred_tr: np.ndarray,
    y_va: np.ndarray,
    pred_va: np.ndarray,
) -> None:
    h = history.history
    mae_tr, rmse_tr = mae_rmse_per_target(y_tr, pred_tr)
    mae_va, rmse_va = mae_rmse_per_target(y_va, pred_va)

    fig = plt.figure(figsize=(11, 8))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.0])
    ax_l = fig.add_subplot(gs[0, 0])
    ax_l.plot(h["loss"], label="train")
    ax_l.plot(h["val_loss"], label="val")
    ax_l.set_title("Loss (MSE)")
    ax_l.set_xlabel("Epoch")
    ax_l.legend(fontsize=8)
    ax_l.grid(alpha=0.3)

    ax_m = fig.add_subplot(gs[0, 1])
    if "mae" in h:
        ax_m.plot(h["mae"], label="train MAE")
    if "val_mae" in h:
        ax_m.plot(h["val_mae"], label="val MAE")
    ax_m.set_title("MAE")
    ax_m.set_xlabel("Epoch")
    ax_m.legend(fontsize=8)
    ax_m.grid(alpha=0.3)

    ax_b = fig.add_subplot(gs[1, :])
    x = np.arange(len(TARGET_METRIC_NAMES))
    w = 0.18
    ax_b.bar(x - 1.5 * w, mae_tr, w, label="MAE train")
    ax_b.bar(x - 0.5 * w, mae_va, w, label="MAE val")
    ax_b.bar(x + 0.5 * w, rmse_tr, w, label="RMSE train")
    ax_b.bar(x + 1.5 * w, rmse_va, w, label="RMSE val")
    ax_b.set_xticks(x)
    ax_b.set_xticklabels(TARGET_METRIC_NAMES)
    ax_b.set_ylabel("Erreur fin d’entraînement (cibles norm. [0,1])")
    ax_b.legend(fontsize=7)
    ax_b.set_title("Dernière epoch (poids restaurés si early stopping)")
    ax_b.grid(axis="y", alpha=0.3)

    fig.suptitle(
        f"{prefix} — cluster {cluster_id} EGID {egid} — courbes + écart train/val (sur-apprentissage ?)"
    )
    fig.tight_layout()
    fig.savefig(model_dir / f"{prefix}_{egid}_train_val_diagnostics.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    lstm_meta = {
        "epochs_ran": len(h.get("loss", [])),
        "final_train_loss": h["loss"][-1] if h.get("loss") else None,
        "final_val_loss": h["val_loss"][-1] if h.get("val_loss") else None,
        "train_mae": mae_tr.tolist(),
        "val_mae": mae_va.tolist(),
        "train_rmse": rmse_tr.tolist(),
        "val_rmse": rmse_va.tolist(),
    }
    (model_dir / f"{prefix}_{egid}_training_summary.json").write_text(
        json.dumps(lstm_meta, indent=2, default=str), encoding="utf-8"
    )


In [ ]:
def sample_egids_all_clusters() -> dict[int, list[str]]:
    selected: dict[int, list[str]] = {}
    for cid in CLUSTER_IDS:
        cols = pq.ParquetFile(PATH_TRAIN / f"cluster{cid}.parquet").schema.names
        all_e = parse_egids(cols)
        if not all_e:
            raise ValueError(f"Cluster {cid}: aucun EGID (colonnes *.TempRet_norm)")
        use_all = (N_EGID_PER_CLUSTER == -1) or (N_EGID_PER_CLUSTER > len(all_e))
        if use_all:
            selected[cid] = list(all_e)
        else:
            pick = RNG.choice(np.array(all_e, dtype=object), size=N_EGID_PER_CLUSTER, replace=False)
            selected[cid] = sorted(str(x) for x in pick.tolist())
    return selected


SELECTED_EGIDS = sample_egids_all_clusters()
for k, v in SELECTED_EGIDS.items():
    print(f"Cluster {k}: {v}")


## 2. Random Forest — un bloc par cluster (EGID selon `N_EGID_PER_CLUSTER`, grille HP sur validation, refit train)

Par EGID : graphique **train vs validation** (MAE / RMSE sur cibles normées), **top 5 importances** de features, et **récapitulatif JSON** des scores de grille + paramètres retenus (`RF_{EGID}_train_val_diagnostics.png`, `RF_{EGID}_training_summary.json`).

In [ ]:
CLUSTER_ID = 3
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("RF cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    best_score = np.inf
    best_params = None
    grid_results: list[tuple[dict, float]] = []
    for params in RF_PARAM_GRID:
        rf = RandomForestRegressor(**params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
        rf.fit(X_tr, y_tr)
        pred_va_grid = rf.predict(X_va)
        sc = aggregate_score(rmse_per_target(y_va, pred_va_grid))
        grid_results.append((dict(params), float(sc)))
        if sc < best_score:
            best_score = sc
            best_params = params

    rf_final = RandomForestRegressor(**best_params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
    rf_final.fit(X_tr, y_tr)
    pred_tr_rf = rf_final.predict(X_tr)
    pred_va_rf = rf_final.predict(X_va)
    pred_te = rf_final.predict(X_te)
    plot_rf_diagnostics(
        model_dir,
        "RF",
        egid,
        CLUSTER_ID,
        y_tr,
        pred_tr_rf,
        y_va,
        pred_va_rf,
        list(FEATURE_COLS),
        rf_final,
        best_params,
        grid_results,
    )

    bundle = {
        "kind": "RF",
        "model": rf_final,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "best_params": best_params,
        "val_rmse_mean_targets": best_score,
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    joblib.dump(bundle, model_dir / f"RF_{egid}.joblib")
    export_test_predictions(result_dir / f"RF_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, rf_final, X_tr, y_tr, X_va, y_va, X_te, y_te, pred_te, bundle, inv_sub, inv_te)

logger.info("RF terminé cluster %s", CLUSTER_ID)
free_ram(egids)


In [ ]:
CLUSTER_ID = 4
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("RF cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    best_score = np.inf
    best_params = None
    grid_results: list[tuple[dict, float]] = []
    for params in RF_PARAM_GRID:
        rf = RandomForestRegressor(**params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
        rf.fit(X_tr, y_tr)
        pred_va_grid = rf.predict(X_va)
        sc = aggregate_score(rmse_per_target(y_va, pred_va_grid))
        grid_results.append((dict(params), float(sc)))
        if sc < best_score:
            best_score = sc
            best_params = params

    rf_final = RandomForestRegressor(**best_params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
    rf_final.fit(X_tr, y_tr)
    pred_tr_rf = rf_final.predict(X_tr)
    pred_va_rf = rf_final.predict(X_va)
    pred_te = rf_final.predict(X_te)
    plot_rf_diagnostics(
        model_dir,
        "RF",
        egid,
        CLUSTER_ID,
        y_tr,
        pred_tr_rf,
        y_va,
        pred_va_rf,
        list(FEATURE_COLS),
        rf_final,
        best_params,
        grid_results,
    )

    bundle = {
        "kind": "RF",
        "model": rf_final,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "best_params": best_params,
        "val_rmse_mean_targets": best_score,
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    joblib.dump(bundle, model_dir / f"RF_{egid}.joblib")
    export_test_predictions(result_dir / f"RF_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, rf_final, X_tr, y_tr, X_va, y_va, X_te, y_te, pred_te, bundle, inv_sub, inv_te)

logger.info("RF terminé cluster %s", CLUSTER_ID)
free_ram(egids)


In [ ]:
CLUSTER_ID = 5
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("RF cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    best_score = np.inf
    best_params = None
    grid_results: list[tuple[dict, float]] = []
    for params in RF_PARAM_GRID:
        rf = RandomForestRegressor(**params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
        rf.fit(X_tr, y_tr)
        pred_va_grid = rf.predict(X_va)
        sc = aggregate_score(rmse_per_target(y_va, pred_va_grid))
        grid_results.append((dict(params), float(sc)))
        if sc < best_score:
            best_score = sc
            best_params = params

    rf_final = RandomForestRegressor(**best_params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
    rf_final.fit(X_tr, y_tr)
    pred_tr_rf = rf_final.predict(X_tr)
    pred_va_rf = rf_final.predict(X_va)
    pred_te = rf_final.predict(X_te)
    plot_rf_diagnostics(
        model_dir,
        "RF",
        egid,
        CLUSTER_ID,
        y_tr,
        pred_tr_rf,
        y_va,
        pred_va_rf,
        list(FEATURE_COLS),
        rf_final,
        best_params,
        grid_results,
    )

    bundle = {
        "kind": "RF",
        "model": rf_final,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "best_params": best_params,
        "val_rmse_mean_targets": best_score,
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    joblib.dump(bundle, model_dir / f"RF_{egid}.joblib")
    export_test_predictions(result_dir / f"RF_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, rf_final, X_tr, y_tr, X_va, y_va, X_te, y_te, pred_te, bundle, inv_sub, inv_te)

logger.info("RF terminé cluster %s", CLUSTER_ID)
free_ram(egids)


In [ ]:
CLUSTER_ID = 6
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("RF cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    best_score = np.inf
    best_params = None
    grid_results: list[tuple[dict, float]] = []
    for params in RF_PARAM_GRID:
        rf = RandomForestRegressor(**params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
        rf.fit(X_tr, y_tr)
        pred_va_grid = rf.predict(X_va)
        sc = aggregate_score(rmse_per_target(y_va, pred_va_grid))
        grid_results.append((dict(params), float(sc)))
        if sc < best_score:
            best_score = sc
            best_params = params

    rf_final = RandomForestRegressor(**best_params, random_state=SEED, n_jobs=N_JOBS_PARALLEL)
    rf_final.fit(X_tr, y_tr)
    pred_tr_rf = rf_final.predict(X_tr)
    pred_va_rf = rf_final.predict(X_va)
    pred_te = rf_final.predict(X_te)
    plot_rf_diagnostics(
        model_dir,
        "RF",
        egid,
        CLUSTER_ID,
        y_tr,
        pred_tr_rf,
        y_va,
        pred_va_rf,
        list(FEATURE_COLS),
        rf_final,
        best_params,
        grid_results,
    )

    bundle = {
        "kind": "RF",
        "model": rf_final,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "best_params": best_params,
        "val_rmse_mean_targets": best_score,
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    joblib.dump(bundle, model_dir / f"RF_{egid}.joblib")
    export_test_predictions(result_dir / f"RF_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, rf_final, X_tr, y_tr, X_va, y_va, X_te, y_te, pred_te, bundle, inv_sub, inv_te)

logger.info("RF terminé cluster %s", CLUSTER_ID)
free_ram(egids)


## 3. XGBoost — un bloc par cluster (early stopping sur validation)

Par EGID : même principe + **top 5 features** (gain XGB moyenné sur les 2 cibles) et **best_iteration** par cible dans le JSON (`XB_*`).

In [ ]:
import xgboost as xgb


In [ ]:
CLUSTER_ID = 3
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("XGB cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    models = {}
    preds_tr = np.zeros_like(y_tr)
    preds_va = np.zeros_like(y_va)
    preds_te = np.zeros_like(y_te)
    for j, name in enumerate(["TempRet", "PuisCpt"]):
        m = xgb.XGBRegressor(
            **XGB_PARAMS,
            callbacks=[xgb.callback.EarlyStopping(rounds=XGB_EARLY_STOPPING_ROUNDS)],
        )
        m.fit(
            X_tr,
            y_tr[:, j],
            eval_set=[(X_va, y_va[:, j])],
            verbose=False,
        )
        models[name] = m
        preds_tr[:, j] = m.predict(X_tr)
        preds_va[:, j] = m.predict(X_va)
        preds_te[:, j] = m.predict(X_te)

    plot_xgb_diagnostics(
        model_dir,
        "XB",
        egid,
        CLUSTER_ID,
        y_tr,
        preds_tr,
        y_va,
        preds_va,
        list(FEATURE_COLS),
        models,
    )

    bundle = {
        "kind": "XGB",
        "models": models,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
        "val_rmse_mean_targets": aggregate_score(rmse_per_target(y_va, preds_va)),
    }
    joblib.dump(bundle, model_dir / f"XB_{egid}.joblib")
    export_test_predictions(result_dir / f"XB_{egid}.parquet", dates_te, y_te, preds_te, inv_te)

    free_ram(X_raw, y, scaler, models, X_tr, y_tr, X_va, y_va, X_te, y_te, preds_te, preds_va, bundle, inv_sub, inv_te)

logger.info("XGB terminé cluster %s", CLUSTER_ID)
free_ram(egids)


In [ ]:
CLUSTER_ID = 4
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("XGB cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    models = {}
    preds_tr = np.zeros_like(y_tr)
    preds_va = np.zeros_like(y_va)
    preds_te = np.zeros_like(y_te)
    for j, name in enumerate(["TempRet", "PuisCpt"]):
        m = xgb.XGBRegressor(
            **XGB_PARAMS,
            callbacks=[xgb.callback.EarlyStopping(rounds=XGB_EARLY_STOPPING_ROUNDS)],
        )
        m.fit(
            X_tr,
            y_tr[:, j],
            eval_set=[(X_va, y_va[:, j])],
            verbose=False,
        )
        models[name] = m
        preds_tr[:, j] = m.predict(X_tr)
        preds_va[:, j] = m.predict(X_va)
        preds_te[:, j] = m.predict(X_te)

    plot_xgb_diagnostics(
        model_dir,
        "XB",
        egid,
        CLUSTER_ID,
        y_tr,
        preds_tr,
        y_va,
        preds_va,
        list(FEATURE_COLS),
        models,
    )

    bundle = {
        "kind": "XGB",
        "models": models,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
        "val_rmse_mean_targets": aggregate_score(rmse_per_target(y_va, preds_va)),
    }
    joblib.dump(bundle, model_dir / f"XB_{egid}.joblib")
    export_test_predictions(result_dir / f"XB_{egid}.parquet", dates_te, y_te, preds_te, inv_te)

    free_ram(X_raw, y, scaler, models, X_tr, y_tr, X_va, y_va, X_te, y_te, preds_te, preds_va, bundle, inv_sub, inv_te)

logger.info("XGB terminé cluster %s", CLUSTER_ID)
free_ram(egids)


In [ ]:
CLUSTER_ID = 5
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("XGB cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    models = {}
    preds_tr = np.zeros_like(y_tr)
    preds_va = np.zeros_like(y_va)
    preds_te = np.zeros_like(y_te)
    for j, name in enumerate(["TempRet", "PuisCpt"]):
        m = xgb.XGBRegressor(
            **XGB_PARAMS,
            callbacks=[xgb.callback.EarlyStopping(rounds=XGB_EARLY_STOPPING_ROUNDS)],
        )
        m.fit(
            X_tr,
            y_tr[:, j],
            eval_set=[(X_va, y_va[:, j])],
            verbose=False,
        )
        models[name] = m
        preds_tr[:, j] = m.predict(X_tr)
        preds_va[:, j] = m.predict(X_va)
        preds_te[:, j] = m.predict(X_te)

    plot_xgb_diagnostics(
        model_dir,
        "XB",
        egid,
        CLUSTER_ID,
        y_tr,
        preds_tr,
        y_va,
        preds_va,
        list(FEATURE_COLS),
        models,
    )

    bundle = {
        "kind": "XGB",
        "models": models,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
        "val_rmse_mean_targets": aggregate_score(rmse_per_target(y_va, preds_va)),
    }
    joblib.dump(bundle, model_dir / f"XB_{egid}.joblib")
    export_test_predictions(result_dir / f"XB_{egid}.parquet", dates_te, y_te, preds_te, inv_te)

    free_ram(X_raw, y, scaler, models, X_tr, y_tr, X_va, y_va, X_te, y_te, preds_te, preds_va, bundle, inv_sub, inv_te)

logger.info("XGB terminé cluster %s", CLUSTER_ID)
free_ram(egids)


In [ ]:
CLUSTER_ID = 6
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("XGB cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_raw[tr_mask])
    y_tr = y[tr_mask]
    X_va = scaler.transform(X_raw[va_mask])
    y_va = y[va_mask]
    X_te = scaler.transform(X_raw[te_mask])
    y_te = y[te_mask]
    dates_te = dates[te_mask]
    inv_te = inv_sub.iloc[np.where(te_mask)[0]].reset_index(drop=True)

    models = {}
    preds_tr = np.zeros_like(y_tr)
    preds_va = np.zeros_like(y_va)
    preds_te = np.zeros_like(y_te)
    for j, name in enumerate(["TempRet", "PuisCpt"]):
        m = xgb.XGBRegressor(
            **XGB_PARAMS,
            callbacks=[xgb.callback.EarlyStopping(rounds=XGB_EARLY_STOPPING_ROUNDS)],
        )
        m.fit(
            X_tr,
            y_tr[:, j],
            eval_set=[(X_va, y_va[:, j])],
            verbose=False,
        )
        models[name] = m
        preds_tr[:, j] = m.predict(X_tr)
        preds_va[:, j] = m.predict(X_va)
        preds_te[:, j] = m.predict(X_te)

    plot_xgb_diagnostics(
        model_dir,
        "XB",
        egid,
        CLUSTER_ID,
        y_tr,
        preds_tr,
        y_va,
        preds_va,
        list(FEATURE_COLS),
        models,
    )

    bundle = {
        "kind": "XGB",
        "models": models,
        "scaler": scaler,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
        "val_rmse_mean_targets": aggregate_score(rmse_per_target(y_va, preds_va)),
    }
    joblib.dump(bundle, model_dir / f"XB_{egid}.joblib")
    export_test_predictions(result_dir / f"XB_{egid}.parquet", dates_te, y_te, preds_te, inv_te)

    free_ram(X_raw, y, scaler, models, X_tr, y_tr, X_va, y_va, X_te, y_te, preds_te, preds_va, bundle, inv_sub, inv_te)

logger.info("XGB terminé cluster %s", CLUSTER_ID)
free_ram(egids)


## 4. LSTM — un bloc par cluster (early stopping Keras)

Par EGID : **courbes loss / MAE** (train vs val) pour repérer sur-apprentissage ou architecture inadaptée, barres d’erreur finales train/val (`LSTM_*`).

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Dense, Dropout


def build_lstm_model(seq_len: int, n_features: int):
    """Architecture LSTM multi-sortie : Dense(2, sigmoid) = [TempRet_norm, PuisCpt_fc] dans [0,1].
    Modifier **uniquement** ce corps pour les 4 blocs cluster."""
    return Sequential(
        [
            LSTM(32, return_sequences=True, input_shape=(seq_len, n_features)), #  64
            Dropout(0.2),
            LSTM(16, return_sequences=False), #  32
            Dropout(0.2),
            Dense(2, activation="sigmoid"),
        ]
    )


def compile_lstm_model(model) -> None:
    """Compile avec optimiseur Adam explicite (taux dans la cellule Configuration)."""
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LSTM_ADAM_LEARNING_RATE),
        loss=LSTM_COMPILE_LOSS,
        metrics=["mae"],
    )


In [ ]:
CLUSTER_ID = 3
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("LSTM cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_scaled = np.empty_like(X_raw, dtype=np.float32)
    X_scaled[tr_mask] = scaler.fit_transform(X_raw[tr_mask]).astype(np.float32)
    X_scaled[~tr_mask] = scaler.transform(X_raw[~tr_mask]).astype(np.float32)

    train_e, val_e, test_e = collect_lstm_end_indices(sp, SEQ_LEN)
    train_e = [t for t in train_e if inv_ok[t]]
    val_e = [t for t in val_e if inv_ok[t]]

    X_seq_tr = ends_to_seq_X(X_scaled, train_e, SEQ_LEN).astype(np.float32)
    y_seq_tr = y[train_e].astype(np.float32)
    X_seq_va = ends_to_seq_X(X_scaled, val_e, SEQ_LEN).astype(np.float32)
    y_seq_va = y[val_e].astype(np.float32)
    X_seq_te = ends_to_seq_X(X_scaled, test_e, SEQ_LEN).astype(np.float32)

    y_te = y[test_e]
    dates_te = dates[test_e]
    inv_te = inv_sub.iloc[test_e].reset_index(drop=True)

    n_feat = X_seq_tr.shape[2]
    keras_m = build_lstm_model(SEQ_LEN, n_feat)
    compile_lstm_model(keras_m)
    es = EarlyStopping(monitor="val_loss", patience=LSTM_PATIENCE, restore_best_weights=True)
    hist = keras_m.fit(
        X_seq_tr,
        y_seq_tr,
        validation_data=(X_seq_va, y_seq_va),
        epochs=LSTM_EPOCHS,
        batch_size=LSTM_BATCH_SIZE,
        callbacks=[es],
        verbose=0,
    )
    pred_tr_lstm = keras_m.predict(X_seq_tr, batch_size=LSTM_BATCH_SIZE, verbose=0)
    pred_va_lstm = keras_m.predict(X_seq_va, batch_size=LSTM_BATCH_SIZE, verbose=0)
    pred_te = keras_m.predict(X_seq_te, batch_size=LSTM_BATCH_SIZE, verbose=0)
    plot_lstm_diagnostics(
        model_dir,
        "LSTM",
        egid,
        CLUSTER_ID,
        hist,
        y_seq_tr,
        pred_tr_lstm,
        y_seq_va,
        pred_va_lstm,
    )

    bundle = {
        "kind": "LSTM",
        "keras_model": keras_m,
        "scaler": scaler,
        "seq_len": SEQ_LEN,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    keras_m.save(model_dir / f"LSTM_{egid}.keras", overwrite=True)
    joblib.dump({k: v for k, v in bundle.items() if k != "keras_model"}, model_dir / f"LSTM_{egid}_meta.joblib")

    export_test_predictions(result_dir / f"LSTM_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, keras_m, X_seq_tr, y_seq_tr, X_seq_va, y_seq_va, X_seq_te, pred_te, bundle, inv_sub, inv_te)
    free_tf()

logger.info("LSTM terminé cluster %s", CLUSTER_ID)
free_ram(egids)


In [ ]:
CLUSTER_ID = 4
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("LSTM cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_scaled = np.empty_like(X_raw, dtype=np.float32)
    X_scaled[tr_mask] = scaler.fit_transform(X_raw[tr_mask]).astype(np.float32)
    X_scaled[~tr_mask] = scaler.transform(X_raw[~tr_mask]).astype(np.float32)

    train_e, val_e, test_e = collect_lstm_end_indices(sp, SEQ_LEN)
    train_e = [t for t in train_e if inv_ok[t]]
    val_e = [t for t in val_e if inv_ok[t]]

    X_seq_tr = ends_to_seq_X(X_scaled, train_e, SEQ_LEN).astype(np.float32)
    y_seq_tr = y[train_e].astype(np.float32)
    X_seq_va = ends_to_seq_X(X_scaled, val_e, SEQ_LEN).astype(np.float32)
    y_seq_va = y[val_e].astype(np.float32)
    X_seq_te = ends_to_seq_X(X_scaled, test_e, SEQ_LEN).astype(np.float32)

    y_te = y[test_e]
    dates_te = dates[test_e]
    inv_te = inv_sub.iloc[test_e].reset_index(drop=True)

    n_feat = X_seq_tr.shape[2]
    keras_m = build_lstm_model(SEQ_LEN, n_feat)
    compile_lstm_model(keras_m)
    es = EarlyStopping(monitor="val_loss", patience=LSTM_PATIENCE, restore_best_weights=True)
    hist = keras_m.fit(
        X_seq_tr,
        y_seq_tr,
        validation_data=(X_seq_va, y_seq_va),
        epochs=LSTM_EPOCHS,
        batch_size=LSTM_BATCH_SIZE,
        callbacks=[es],
        verbose=0,
    )
    pred_tr_lstm = keras_m.predict(X_seq_tr, batch_size=LSTM_BATCH_SIZE, verbose=0)
    pred_va_lstm = keras_m.predict(X_seq_va, batch_size=LSTM_BATCH_SIZE, verbose=0)
    pred_te = keras_m.predict(X_seq_te, batch_size=LSTM_BATCH_SIZE, verbose=0)
    plot_lstm_diagnostics(
        model_dir,
        "LSTM",
        egid,
        CLUSTER_ID,
        hist,
        y_seq_tr,
        pred_tr_lstm,
        y_seq_va,
        pred_va_lstm,
    )

    bundle = {
        "kind": "LSTM",
        "keras_model": keras_m,
        "scaler": scaler,
        "seq_len": SEQ_LEN,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    keras_m.save(model_dir / f"LSTM_{egid}.keras", overwrite=True)
    joblib.dump({k: v for k, v in bundle.items() if k != "keras_model"}, model_dir / f"LSTM_{egid}_meta.joblib")

    export_test_predictions(result_dir / f"LSTM_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, keras_m, X_seq_tr, y_seq_tr, X_seq_va, y_seq_va, X_seq_te, pred_te, bundle, inv_sub, inv_te)
    free_tf()

logger.info("LSTM terminé cluster %s", CLUSTER_ID)
free_ram(egids)


In [ ]:
CLUSTER_ID = 5
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("LSTM cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_scaled = np.empty_like(X_raw, dtype=np.float32)
    X_scaled[tr_mask] = scaler.fit_transform(X_raw[tr_mask]).astype(np.float32)
    X_scaled[~tr_mask] = scaler.transform(X_raw[~tr_mask]).astype(np.float32)

    train_e, val_e, test_e = collect_lstm_end_indices(sp, SEQ_LEN)
    train_e = [t for t in train_e if inv_ok[t]]
    val_e = [t for t in val_e if inv_ok[t]]

    X_seq_tr = ends_to_seq_X(X_scaled, train_e, SEQ_LEN).astype(np.float32)
    y_seq_tr = y[train_e].astype(np.float32)
    X_seq_va = ends_to_seq_X(X_scaled, val_e, SEQ_LEN).astype(np.float32)
    y_seq_va = y[val_e].astype(np.float32)
    X_seq_te = ends_to_seq_X(X_scaled, test_e, SEQ_LEN).astype(np.float32)

    y_te = y[test_e]
    dates_te = dates[test_e]
    inv_te = inv_sub.iloc[test_e].reset_index(drop=True)

    n_feat = X_seq_tr.shape[2]
    keras_m = build_lstm_model(SEQ_LEN, n_feat)
    compile_lstm_model(keras_m)
    es = EarlyStopping(monitor="val_loss", patience=LSTM_PATIENCE, restore_best_weights=True)
    hist = keras_m.fit(
        X_seq_tr,
        y_seq_tr,
        validation_data=(X_seq_va, y_seq_va),
        epochs=LSTM_EPOCHS,
        batch_size=LSTM_BATCH_SIZE,
        callbacks=[es],
        verbose=0,
    )
    pred_tr_lstm = keras_m.predict(X_seq_tr, batch_size=LSTM_BATCH_SIZE, verbose=0)
    pred_va_lstm = keras_m.predict(X_seq_va, batch_size=LSTM_BATCH_SIZE, verbose=0)
    pred_te = keras_m.predict(X_seq_te, batch_size=LSTM_BATCH_SIZE, verbose=0)
    plot_lstm_diagnostics(
        model_dir,
        "LSTM",
        egid,
        CLUSTER_ID,
        hist,
        y_seq_tr,
        pred_tr_lstm,
        y_seq_va,
        pred_va_lstm,
    )

    bundle = {
        "kind": "LSTM",
        "keras_model": keras_m,
        "scaler": scaler,
        "seq_len": SEQ_LEN,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    keras_m.save(model_dir / f"LSTM_{egid}.keras", overwrite=True)
    joblib.dump({k: v for k, v in bundle.items() if k != "keras_model"}, model_dir / f"LSTM_{egid}_meta.joblib")

    export_test_predictions(result_dir / f"LSTM_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, keras_m, X_seq_tr, y_seq_tr, X_seq_va, y_seq_va, X_seq_te, pred_te, bundle, inv_sub, inv_te)
    free_tf()

logger.info("LSTM terminé cluster %s", CLUSTER_ID)
free_ram(egids)


In [ ]:
CLUSTER_ID = 6
egids = SELECTED_EGIDS[CLUSTER_ID]
model_dir, result_dir = ensure_dirs(CLUSTER_ID)

for egid in egids:
    logger.info("LSTM cluster %s EGID %s", CLUSTER_ID, egid)
    full, _sizes = load_concat_frames(CLUSTER_ID, egid)
    X_raw, y, sp, inv_ok, dates, inv_sub = build_xy_matrices(full, egid)
    free_ram(full)

    tr_mask = (sp == 0) & inv_ok
    va_mask = (sp == 1) & inv_ok
    te_mask = sp == 2

    scaler = StandardScaler()
    X_scaled = np.empty_like(X_raw, dtype=np.float32)
    X_scaled[tr_mask] = scaler.fit_transform(X_raw[tr_mask]).astype(np.float32)
    X_scaled[~tr_mask] = scaler.transform(X_raw[~tr_mask]).astype(np.float32)

    train_e, val_e, test_e = collect_lstm_end_indices(sp, SEQ_LEN)
    train_e = [t for t in train_e if inv_ok[t]]
    val_e = [t for t in val_e if inv_ok[t]]

    X_seq_tr = ends_to_seq_X(X_scaled, train_e, SEQ_LEN).astype(np.float32)
    y_seq_tr = y[train_e].astype(np.float32)
    X_seq_va = ends_to_seq_X(X_scaled, val_e, SEQ_LEN).astype(np.float32)
    y_seq_va = y[val_e].astype(np.float32)
    X_seq_te = ends_to_seq_X(X_scaled, test_e, SEQ_LEN).astype(np.float32)

    y_te = y[test_e]
    dates_te = dates[test_e]
    inv_te = inv_sub.iloc[test_e].reset_index(drop=True)

    n_feat = X_seq_tr.shape[2]
    keras_m = build_lstm_model(SEQ_LEN, n_feat)
    compile_lstm_model(keras_m)
    es = EarlyStopping(monitor="val_loss", patience=LSTM_PATIENCE, restore_best_weights=True)
    hist = keras_m.fit(
        X_seq_tr,
        y_seq_tr,
        validation_data=(X_seq_va, y_seq_va),
        epochs=LSTM_EPOCHS,
        batch_size=LSTM_BATCH_SIZE,
        callbacks=[es],
        verbose=0,
    )
    pred_tr_lstm = keras_m.predict(X_seq_tr, batch_size=LSTM_BATCH_SIZE, verbose=0)
    pred_va_lstm = keras_m.predict(X_seq_va, batch_size=LSTM_BATCH_SIZE, verbose=0)
    pred_te = keras_m.predict(X_seq_te, batch_size=LSTM_BATCH_SIZE, verbose=0)
    plot_lstm_diagnostics(
        model_dir,
        "LSTM",
        egid,
        CLUSTER_ID,
        hist,
        y_seq_tr,
        pred_tr_lstm,
        y_seq_va,
        pred_va_lstm,
    )

    bundle = {
        "kind": "LSTM",
        "keras_model": keras_m,
        "scaler": scaler,
        "seq_len": SEQ_LEN,
        "feature_columns": list(FEATURE_COLS),
        "egid": egid,
        "cluster_id": CLUSTER_ID,
    }
    keras_m.save(model_dir / f"LSTM_{egid}.keras", overwrite=True)
    joblib.dump({k: v for k, v in bundle.items() if k != "keras_model"}, model_dir / f"LSTM_{egid}_meta.joblib")

    export_test_predictions(result_dir / f"LSTM_{egid}.parquet", dates_te, y_te, pred_te, inv_te)

    free_ram(X_raw, y, scaler, keras_m, X_seq_tr, y_seq_tr, X_seq_va, y_seq_va, X_seq_te, pred_te, bundle, inv_sub, inv_te)
    free_tf()

logger.info("LSTM terminé cluster %s", CLUSTER_ID)
free_ram(egids)


## 5. Analyse et comparaison

- Métriques sur les fichiers test exportés (`RF_*`, `XB_*`, `LSTM_*`).
- Meilleur modèle par EGID (score = moyenne des RMSE sur **TempRet en °C** (déjà dénormé dans les parquet) et **PuisCpt en fc** [0,1]).
- Synthèse par cluster et globale ; graphiques comparatifs (plage configurable).


In [ ]:
PREFIXES = ("RF", "XB", "LSTM")


def load_result_metrics(cluster_id: int) -> pd.DataFrame:
    rows = []
    rdir = PATH_RESULTS / f"Cluster{cluster_id}"
    for egid in SELECTED_EGIDS[cluster_id]:
        for pref in PREFIXES:
            p = rdir / f"{pref}_{egid}.parquet"
            if not p.exists():
                logger.warning("Manquant: %s", p)
                continue
            df = pd.read_parquet(p)
            yt = df[["TempRet", "PuisCpt"]].to_numpy()
            yp = df[["TempRetPred", "PuisCptPred"]].to_numpy()
            mae_t = mean_absolute_error(yt[:, 0], yp[:, 0])
            mae_p = mean_absolute_error(yt[:, 1], yp[:, 1])
            rmse_t = np.sqrt(mean_squared_error(yt[:, 0], yp[:, 0]))
            rmse_p = np.sqrt(mean_squared_error(yt[:, 1], yp[:, 1]))
            score = float(np.mean([rmse_t, rmse_p]))
            rows.append(
                dict(
                    cluster_id=cluster_id,
                    egid=egid,
                    model=pref,
                    rmse_TempRet=rmse_t,
                    rmse_PuisCpt=rmse_p,
                    mae_TempRet=mae_t,
                    mae_PuisCpt=mae_p,
                    score=score,
                )
            )
    return pd.DataFrame(rows)


metrics_all = pd.concat([load_result_metrics(c) for c in CLUSTER_IDS], ignore_index=True)
metrics_all.to_csv(PATH_RESULTS / "metrics_all_models.csv", index=False)
print(metrics_all.groupby("model")["score"].mean())

best_per_egid = metrics_all.sort_values("score").groupby(["cluster_id", "egid"], as_index=False).first()
best_per_egid.to_csv(PATH_RESULTS / "best_model_per_egid.csv", index=False)
print("Meilleur modèle par EGID (extrait):")
print(best_per_egid.head(20))

best_by_cluster = best_per_egid.groupby("cluster_id")["model"].agg(lambda x: x.mode().iloc[0] if len(x) else None)
print("Modèle dominant par cluster (mode des gagnants EGID):")
print(best_by_cluster)

global_counts = best_per_egid["model"].value_counts()
print("Répartition globale des meilleurs modèles par EGID:")
print(global_counts)
best_global = global_counts.index[0] if len(global_counts) else None
print("Meilleur type global (vote EGID):", best_global)


In [ ]:
import matplotlib.pyplot as plt


def plot_egid_comparison(cluster_id: int, egid: str) -> None:
    rdir = PATH_RESULTS / f"Cluster{cluster_id}"
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    for pref, style in [("RF", "-"), ("XB", "--"), ("LSTM", ":")]:
        p = rdir / f"{pref}_{egid}.parquet"
        if not p.exists():
            continue
        df = pd.read_parquet(p)
        d = df.copy()
        d["Dates"] = pd.to_datetime(d["Dates"], utc=True)
        if PLOT_START is not None:
            d = d[d["Dates"] >= pd.Timestamp(PLOT_START)]
        if PLOT_END is not None:
            d = d[d["Dates"] <= pd.Timestamp(PLOT_END)]
        if PLOT_START is None and PLOT_END is None and PLOT_MAX_POINTS and len(d) > PLOT_MAX_POINTS:
            d = d.iloc[-PLOT_MAX_POINTS :]
        ax0, ax1 = axes[0], axes[1]
        ax0.plot(d["Dates"], d["TempRet"], color="k", alpha=0.35, linewidth=1, label="_nolegend_" if pref != "RF" else "cible")
        ax0.plot(d["Dates"], d["TempRetPred"], linestyle=style, linewidth=1.2, label=f"{pref} pred")
        ax1.plot(d["Dates"], d["PuisCpt"], color="k", alpha=0.35, linewidth=1, label="_nolegend_" if pref != "RF" else "cible")
        ax1.plot(d["Dates"], d["PuisCptPred"], linestyle=style, linewidth=1.2, label=f"{pref} pred")
    axes[0].set_ylabel("TempRet (°C)")
    axes[1].set_ylabel("PuisCpt (fc)")
    axes[0].set_title(f"Cluster {cluster_id} — EGID {egid}")
    axes[0].legend(loc="upper right", fontsize=8)
    axes[1].legend(loc="upper right", fontsize=8)
    fig.autofmt_xdate()
    fig.tight_layout()
    out = rdir / f"compare_{egid}_TempRet_PuisCpt.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    logger.info("Graphique %s", out)


for cid in CLUSTER_IDS:
    for egid in SELECTED_EGIDS[cid]:
        plot_egid_comparison(cid, egid)


## 6. Export récapitulatif global (CSV)

Fichier **`0_results_YYYYMMDD_HHmm.CSV`** dans `0_Data/9_Results/` : par EGID entraîné, classement des 3 meilleurs modèles (critère : score test = moyenne des RMSE), hyperparamètres retenus et métriques issues des résumés d’entraînement (`6_Models/Cluster*/`*`_training_summary.json`) ainsi que métriques **test** agrégées.

In [ ]:
from datetime import datetime


def _mean_list(d: dict | None, key: str) -> float | str:
    if d is None or key not in d or not d[key]:
        return ""
    return float(np.mean(d[key]))


def _load_json(path: Path) -> dict | None:
    if not path.is_file():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        logger.warning("JSON illisible %s: %s", path, e)
        return None


def _row_training_rf(mdir: Path, egid: str) -> dict:
    d = _load_json(mdir / f"RF_{egid}_training_summary.json")
    if d is None:
        return {
            "RF_hyperparams_best": "",
            "RF_mae_train_mean": "",
            "RF_rmse_train_mean": "",
            "RF_mae_val_mean": "",
            "RF_rmse_val_mean": "",
        }
    return {
        "RF_hyperparams_best": json.dumps(d.get("best_params"), ensure_ascii=False),
        "RF_mae_train_mean": _mean_list(d, "train_mae"),
        "RF_rmse_train_mean": _mean_list(d, "train_rmse"),
        "RF_mae_val_mean": _mean_list(d, "val_mae"),
        "RF_rmse_val_mean": _mean_list(d, "val_rmse"),
    }


def _row_training_xb(mdir: Path, egid: str) -> dict:
    d = _load_json(mdir / f"XB_{egid}_training_summary.json")
    if d is None:
        return {
            "XB_hyperparams_best": "",
            "XB_mae_train_mean": "",
            "XB_rmse_train_mean": "",
            "XB_mae_val_mean": "",
            "XB_rmse_val_mean": "",
        }
    hp = {k: d[k] for k in ("TempRet", "PuisCpt") if k in d and isinstance(d[k], dict)}
    return {
        "XB_hyperparams_best": json.dumps(hp, ensure_ascii=False),
        "XB_mae_train_mean": _mean_list(d, "train_mae"),
        "XB_rmse_train_mean": _mean_list(d, "train_rmse"),
        "XB_mae_val_mean": _mean_list(d, "val_mae"),
        "XB_rmse_val_mean": _mean_list(d, "val_rmse"),
    }


def _row_training_lstm(mdir: Path, egid: str) -> dict:
    d = _load_json(mdir / f"LSTM_{egid}_training_summary.json")
    if d is None:
        return {
            "LSTM_hyperparams_run": "",
            "LSTM_mae_train_mean": "",
            "LSTM_rmse_train_mean": "",
            "LSTM_mae_val_mean": "",
            "LSTM_rmse_val_mean": "",
        }
    hp = {k: d[k] for k in ("epochs_ran", "final_train_loss", "final_val_loss") if k in d}
    hp["SEQ_LEN"] = SEQ_LEN
    hp["LSTM_EPOCHS_max"] = LSTM_EPOCHS
    return {
        "LSTM_hyperparams_run": json.dumps(hp, ensure_ascii=False),
        "LSTM_mae_train_mean": _mean_list(d, "train_mae"),
        "LSTM_rmse_train_mean": _mean_list(d, "train_rmse"),
        "LSTM_mae_val_mean": _mean_list(d, "val_mae"),
        "LSTM_rmse_val_mean": _mean_list(d, "val_rmse"),
    }


def _row_test_metrics(metrics_df: pd.DataFrame, cid: int, egid: str, pref: str) -> dict:
    sub = metrics_df[
        (metrics_df["cluster_id"] == cid) & (metrics_df["egid"].astype(str) == str(egid)) & (metrics_df["model"] == pref)
    ]
    if sub.empty:
        return {f"{pref}_test_mae_mean": "", f"{pref}_test_rmse_mean": ""}
    r = sub.iloc[0]
    return {
        f"{pref}_test_mae_mean": float((r["mae_TempRet"] + r["mae_PuisCpt"]) / 2),
        f"{pref}_test_rmse_mean": float((r["rmse_TempRet"] + r["rmse_PuisCpt"]) / 2),
    }


def _top3_models(metrics_df: pd.DataFrame, cid: int, egid: str) -> tuple[str, str, str]:
    sub = metrics_df[
        (metrics_df["cluster_id"] == cid) & (metrics_df["egid"].astype(str) == str(egid))
    ].sort_values("score")
    names = sub["model"].tolist()
    out = names[:3] + [""] * 3
    return out[0], out[1], out[2]


_metrics_export = pd.concat([load_result_metrics(c) for c in CLUSTER_IDS], ignore_index=True)
_ts = datetime.now().strftime("%Y%m%d_%H%M")
_out_csv = PATH_RESULTS / f"0_results_{_ts}.CSV"

_rows = []
for cid in CLUSTER_IDS:
    mdir = PATH_MODELS / f"Cluster{cid}"
    for egid in SELECTED_EGIDS[cid]:
        r1, r2, r3 = _top3_models(_metrics_export, cid, egid)
        row = {
            "date_heure_generation": datetime.now().isoformat(timespec="seconds"),
            "cluster_id": cid,
            "egid": str(egid),
            "meilleur_modele": r1,
            "deuxieme_meilleur_modele": r2,
            "troisieme_meilleur_modele": r3,
        }
        row.update(_row_training_rf(mdir, egid))
        row.update(_row_training_xb(mdir, egid))
        row.update(_row_training_lstm(mdir, egid))
        for pref in PREFIXES:
            row.update(_row_test_metrics(_metrics_export, cid, egid, pref))
        _rows.append(row)

pd.DataFrame(_rows).to_csv(_out_csv, index=False, encoding="utf-8-sig")
logger.info("Export récapitulatif: %s (%s lignes)", _out_csv, len(_rows))
print(_out_csv)


## Annexe — Noms des features (exogènes + dérivées)

Les modèles tabulaires (RF, XGB) et le LSTM utilisent un vecteur de features par pas de temps. Outre **`EXOG_COLS`** (voir cellule *Configuration* : `TempExt_norm`, encodages cycliques `dayofyear_*`, `dayofweek_*`, `hour_*`), le notebook ajoute des **colonnes dérivées** par EGID, construites dans `add_lag_features` à partir des cibles normalisées du parquet :

- **`{EGID}.TempRet_norm`** → préfixe **`tr`** dans les noms synthétiques ci-dessous  
- **`{EGID}.PuisCpt_fc`** → préfixe **`pc`**

Les décalages et fenêtres utilisent les listes **`LAGS`** et **`ROLL_WINDOWS`** de la configuration (par défaut `LAGS = [1, 2, 3, 7]`, `ROLL_WINDOWS = [3, 7]`). Un pas = un intervalle de la grille temporelle du dataset (ex. 15 min si le pipeline est en 15 min).

### Lags (`lag_tr_*`, `lag_pc_*`)

| Colonne | Signification |
|---------|----------------|
| **`lag_tr_N`** | Valeur de **`TempRet_norm`** à **N pas** en arrière : `shift(N)` (pas de fuite : on n’utilise pas la valeur au même instant que la cible prédite). |
| **`lag_pc_N`** | Idem pour **`PuisCpt_fc`**. |

Pour chaque `N` dans **`LAGS`**, une paire `lag_tr_N` / `lag_pc_N` est créée.

### Moyennes et écarts-types glissants (`roll_*`)

Les statistiques roulantes sont calculées sur la série **déjà décalée d’un pas** (`shift(1)`) pour limiter la fuite d’information vers la cible au même timestamp.

| Colonne | Signification |
|---------|----------------|
| **`roll_tr_mean_W`** | Moyenne de **`TempRet_norm`** sur les **W** derniers pas **strictement avant** l’instant courant (fenêtre glissante). |
| **`roll_tr_std_W`** | Écart-type de **`TempRet_norm`** sur la même fenêtre. |
| **`roll_pc_mean_W`** | Moyenne de **`PuisCpt_fc`** sur **W** pas. |
| **`roll_pc_std_W`** | Écart-type de **`PuisCpt_fc`** sur **W** pas. |

Pour chaque `W` dans **`ROLL_WINDOWS`**, ces quatre colonnes sont ajoutées.

### Lecture des graphiques d’importance (RF / XGB)

Les figures **« Top 5 features »** listent des noms du type `lag_tr_1` ou `roll_pc_mean_7` : ce sont bien ces colonnes dérivées (souvent très informatives pour la prédiction), en plus des variables d’`EXOG_COLS` lorsque celles-ci apparaissent dans le classement.
